# Forecast exploration

`df`     — long: one row per (city, date, model, lead_day) with forecast + truth + error
`wide`   — one row per (city, date, lead_day), one column per model
`hourly` — raw hourly forecast curves

Built by `fetch_forecasts.py` → `build_dataset.py`. Nothing here refetches.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

ROOT = "data/processed"
df     = pd.read_parquet(f"{ROOT}/dataset/daily_long.parquet")
wide   = pd.read_parquet(f"{ROOT}/dataset/daily_wide.parquet")
hourly = pd.read_parquet(f"{ROOT}/forecasts/hourly_temp.parquet")

df["date"]   = pd.to_datetime(df["date"])
wide["date"] = pd.to_datetime(wide["date"])
hourly["valid_time_utc"] = pd.to_datetime(hourly["valid_time_utc"], utc=True)

# gfs_hrrr duplicates gfs_seamless at leads 0-1 (seamless stitches HRRR in).
DUPES = ["gfs_hrrr"]
df   = df[~df.model.isin(DUPES)]
wide = wide.drop(columns=[f"fc_{m}" for m in DUPES], errors="ignore")

print(df.shape, wide.shape, hourly.shape)
df.head()

## Columns

In [ ]:
print("MODELS :", sorted(df.model.unique()))
print("CITIES :", sorted(df.city.unique()))
print("LEADS  :", sorted(df.lead_day.unique()))
print("DATES  :", df.date.min().date(), "->", df.date.max().date(), f"({df.date.nunique()} days)")
print()
for c, t in df.dtypes.items():
    print(f"  {c:18} {str(t):22}")

### What the columns mean

| | |
|---|---|
| `fc_max_f` | model's daily max, over the **local** calendar day |
| `fc_peak_hour` | local hour of that max |
| `fc_w` | plateau width — hours within 1°F of the max |
| `fc_a` | asymmetry — `T(peak+3h) − T(peak−3h)`; positive = slow cooling |
| `T_cli` / `H_cli` | CLI settlement truth: max °F, local peak hour |
| `e_T` | `T_cli − fc_max_f` — **positive = forecast ran cold** |
| `e_H` | wrapped `H_cli − fc_peak_hour`, in [−12, +12) |

`lead_day = N` means forecast issued N days ahead. Lead 0 is excluded (it's analysis, not a forecast).

## The model matrix — `m` for one day

In [ ]:
FC = [c for c in wide.columns if c.startswith("fc_")]

day = wide[(wide.city == "new_york") & (wide.lead_day == 1)].set_index("date")
day[FC + ["ens_mean", "ens_sd", "T_cli"]].head(15).round(1)

## Pivots you'll want

In [ ]:
def matrix(city, lead=1, value="fc_max_f"):
    """rows = date, cols = model."""
    s = df[(df.city == city) & (df.lead_day == lead)]
    return s.pivot_table(index="date", columns="model", values=value)

def errors(city, lead=1):
    """e_i = T_cli - m_i.  Rows = date, cols = model."""
    return matrix(city, lead, "e_T")

def truth(city):
    return df[df.city == city].groupby("date").T_cli.first()

E = errors("new_york", 1)
E.head().round(2)

## Error covariance and n_eff

`Σ` is the object the whole inverse problem turns on. Under equicorrelation:

$$n_{\text{eff}} = \frac{n}{1 + (n-1)\bar\rho}$$

Note this uses `e_T` against **CLI truth**, so shared initial-condition error *and* the
grid-vs-station offset are both in there — unlike the lead-0 pseudo-truth version,
which cancelled exactly the term that creates correlation.

In [ ]:
for city in sorted(df.city.unique()):
    for lead in (1, 3):
        E = errors(city, lead).dropna(axis=1, how="all").dropna()
        if E.shape[0] < 10 or E.shape[1] < 2:
            print(f"{city:12} d{lead}  too few complete rows"); continue
        C = E.corr(); n = C.shape[0]
        rho = C.values[~np.eye(n, dtype=bool)].mean()
        neff = n / (1 + (n - 1) * rho)
        print(f"{city:12} d{lead}  n={n}  days={len(E)}  "
              f"rho={rho:.3f}  n_eff={neff:.2f}")

### Full correlation matrix

In [ ]:
E = errors("new_york", 1).dropna(axis=1, how="all").dropna()
E.corr().round(2)

## Bias vector `b`

All nine models share a sign per city. That is not model error — it is the
grid-vs-station offset. Rank on **sd**, not MAE: bias is free to remove, spread is not.

In [ ]:
for city in sorted(df.city.unique()):
    E = errors(city, 1)
    s = pd.DataFrame({"bias": E.mean(), "sd": E.std(),
                      "mae": E.abs().mean(), "n": E.notna().sum()})
    print(f"\n{city}")
    print(s.sort_values("sd").round(2).to_string())

## Leave-one-out bias correction

Never use today's error to correct today. This is the honest C0/C1 comparison:
debias every model out-of-sample, then blend.

In [ ]:
def loo_debias(city, lead=1):
    M = matrix(city, lead)
    T = truth(city).reindex(M.index)
    E = M.sub(T, axis=0)
    b_loo = (E.sum() - E) / (E.notna().sum() - 1)   # exclude own day
    return M - b_loo, T

for city in sorted(df.city.unique()):
    Mc, T = loo_debias(city, 1)
    per = (Mc.sub(T, axis=0)).abs().mean().sort_values()
    ens = (Mc.mean(axis=1) - T).abs().mean()
    print(f"{city:12} best_member={per.iloc[0]:.2f} ({per.index[0]})   "
          f"equal_weight_blend={ens:.2f}   blend_wins_by={per.iloc[0]-ens:+.2f}")

## Skill vs lead

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, city in zip(axes, sorted(df.city.unique())):
    s = df[df.city == city]
    piv = s.pivot_table(index="lead_day", columns="model",
                        values="e_T", aggfunc=lambda x: x.abs().mean())
    piv.plot(ax=ax, marker="o", legend=(city == "new_york"))
    ax.set_title(city); ax.set_xlabel("lead (days)"); ax.grid(alpha=.3)
axes[0].set_ylabel("MAE  (F)")
if axes[-1].get_legend(): axes[-1].get_legend().remove()
axes[0].legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

## Error distribution — Gaussian or heavy-tailed?

In [ ]:
from scipy import stats

fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for j, city in enumerate(sorted(df.city.unique())):
    e = df[(df.city == city) & (df.lead_day == 1)].e_T.dropna()
    e = (e - e.mean()) / e.std()
    axes[0, j].hist(e, bins=25, density=True, alpha=.6)
    x = np.linspace(-4, 4, 200)
    axes[0, j].plot(x, stats.norm.pdf(x), label="N(0,1)")
    axes[0, j].plot(x, stats.t.pdf(x * np.sqrt(4/2), 4) * np.sqrt(4/2), label="t(4)")
    axes[0, j].set_title(f"{city}  kurtosis={stats.kurtosis(e):.2f}")
    axes[0, j].legend(fontsize=8)
    stats.probplot(e, dist="norm", plot=axes[1, j])
    axes[1, j].set_title("")
plt.tight_layout(); plt.show()

## Hourly curves — where `w`, `a` and peak hour come from

In [ ]:
TZ = {"new_york": "America/New_York", "los_angeles": "America/Los_Angeles",
      "miami": "America/New_York"}

def curves(city, date, lead=1):
    h = hourly[(hourly.city == city) & (hourly.lead_day == lead)].copy()
    lt = h.valid_time_utc.dt.tz_convert(TZ[city])
    h["local_date"], h["hour"] = lt.dt.date, lt.dt.hour + lt.dt.minute / 60
    h = h[h.local_date == pd.Timestamp(date).date()]
    return h.pivot_table(index="hour", columns="model", values="temp_f")

d = df[df.city == "new_york"].date.iloc[0]
c = curves("new_york", d)
ax = c.plot(figsize=(11, 5), marker=".", title=f"new_york  {pd.Timestamp(d).date()}  lead 1")
T = df[(df.city=="new_york") & (df.date==d)].T_cli.iloc[0]
ax.axhline(T, ls="--", c="k", label=f"CLI truth {T:.0f}F"); ax.legend(fontsize=8, ncol=2)
ax.set_xlabel("local hour"); ax.set_ylabel("F"); ax.grid(alpha=.3); plt.show()

## Bin probabilities → the tradeable object

Kalshi bins are 2°F. CLI reports integers, so the likelihood is interval-censored:
`P(reported = k) = P(k−0.5 ≤ T < k+0.5)`. Discretise last — keep `T` continuous
through the inference and integrate over bin edges only at the end.

In [ ]:
from scipy import stats

def bin_probs(mu, sigma, edges, dist="norm", nu=4):
    d = stats.norm(mu, sigma) if dist == "norm" else stats.t(nu, mu, sigma)
    return np.diff(d.cdf(edges))

city, lead = "new_york", 1
Mc, T = loo_debias(city, lead)
mu = Mc.mean(axis=1)
sd = (Mc.sub(T, axis=0)).std()          # residual sd after debias
edges = np.arange(60, 106, 2)

d0 = mu.index[0]
p_n = bin_probs(mu.loc[d0], sd.mean(), edges, "norm")
p_t = bin_probs(mu.loc[d0], sd.mean(), edges, "t")
out = pd.DataFrame({"bin": [f"{a}-{b}" for a, b in zip(edges[:-1], edges[1:])],
                    "normal": p_n.round(3), "student_t": p_t.round(3)})
print(f"{city} {d0.date()}  mu={mu.loc[d0]:.1f}F  sd={sd.mean():.2f}F  truth={T.loc[d0]:.0f}F")
out[out.normal > 0.005]

## Next

- **Σ is unbalanced** — GEM d6 and UKMO d3 have multi-day outages, so pairwise-complete
  correlation is computed on different date sets per pair and can go non-positive-definite.
  Use complete cases or shrinkage (Ledoit–Wolf) rather than letting `.corr()` decide.
- `sd` above is pooled across days. Heteroskedastic version: regress `log σ` on `ens_sd`, `fc_w`.
- Market side isn't loaded yet — implied PMF from Kalshi bins is what turns any of this
  into an edge.